# Tess Follow Up Observations 2

This notebook is the second part of a follow-up observation on TESS objects of interest (TOIs). The ultimate goal is to create a list of TOIs observable from Sutherland, South Africa. The list will be used to plan follow-up observations with the SAAO 1.0m telescope. In part 1, I gathered a list of transiting exoplanets which have more than 2 comparisons in the Mookodi field of view. In this notebook, I determine which of these objects have transits which are visible at night from Sutherland during my observing run in mid-February.

In [32]:
# import necessary packages
import numpy as np
import pandas as pd

import astropy.units as u
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time,TimeDelta

from astroplan import Observer
from astroplan import FixedTarget
from astropy.coordinates import EarthLocation
from astropy.time import Time
from astroplan.plots import plot_airmass
from astroplan import EclipsingSystem
from astroplan import (PrimaryEclipseConstraint, is_event_observable,
                       AtNightConstraint, AltitudeConstraint, LocalTimeConstraint)

In [57]:
def astroplan_init(observing_site, ra, dec, id_name, eclipse_time, orbital_period, eclipse_duration, obs_time):
    # base cases
    if orbital_period == 0:
        return 0, ""

    # define observing site
    obs_location = EarthLocation.of_site(observing_site)
    observer = Observer.at_site(observing_site)

    #convert our RA and DEC into an astropy Sky Coordinate
    star_coordinates = SkyCoord(ra, dec, unit="deg")

    # define details of transit time and such
    primary_eclipse_time = Time(eclipse_time, format='jd')
    orbital_period = orbital_period * u.day
    eclipse_duration = eclipse_duration * u.hour

    # let astroplan know the name and location of our target star
    star = FixedTarget(name=id_name, coord=star_coordinates)

    #let astroplan know we have a transiting system
    curr_toi = EclipsingSystem(primary_eclipse_time=primary_eclipse_time,
                           orbital_period=orbital_period, duration=eclipse_duration,
                           name=id_name)

    # the start time of our observing run
    obs_time = Time(obs_time)

    #approximate number of transits which would be visible in a two week (14 day) period
    num_transits = 14.0/orbital_period.value
    n_transits = round(num_transits - 0.5)

    constraints = [AtNightConstraint.twilight_civil(),
               AltitudeConstraint(min=30*u.deg)]

    ing_egr = curr_toi.next_primary_ingress_egress_time(obs_time, n_eclipses=n_transits)

    # using our constraints, determine if both the ingress and egress are observable
    can_observe = is_event_observable(constraints, observer, star, times_ingress_egress=ing_egr)
    num_observable = can_observe.sum()

    can_observeT = can_observe.T
    can_obs = np.insert(can_observeT, 1, can_observeT[:,0], axis=1)
    observe_times = ing_egr[can_obs]

    # Format the transit times as a string, properly escaped for CSV
    if any(observe_times):
        t2 = Time(observe_times, format='isot')
        # Convert times to list and join with semicolons instead of spaces
        time_str = ';'.join([t.split('.')[0] for t in t2.iso])  # Remove fractional seconds
    else:
        time_str = ""

    return num_observable, time_str

In [58]:
# reading csv file created in part 1
toi = pd.read_csv('../Tables/TESS_TOI_28Jan2025_Mookodi.csv')
toi.head()

,TIC ID,TOI,Previous CTOI,Master,SG1A,SG1B,SG2,SG3,SG4,SG5,...,Stellar Mass (M_Sun) err,Sectors,Date TOI Alerted (UTC),Date TOI Updated (UTC),Date Modified,Comments,RA (deg),Dec (deg),Mookodi Comp,SHOC Comp
0,231663901,101.01,NaN,5,5,5,5,5,5,5,...,0.129454,"1,27,67",2018-09-05,2024-09-06,2024-10-01 12:15:24,WASP-46 b,318.737000,-55.871864,17,2
1,336732616,103.01,NaN,5,5,5,5,5,5,5,...,0.196969,1,2018-09-05,2024-09-06,2024-09-07 00:00:00,HATS-3 b,312.457500,-24.428694,11,3
2,289793076,108.01,NaN,5,5,5,5,5,5,5,...,NaN,"1,28",2018-09-05,2021-10-20,2022-12-14 12:09:24,HATS-13 b,316.961500,-26.096719,19,1
3,29344935,109.01,NaN,5,5,5,5,5,5,5,...,NaN,"1,28,68",2018-09-05,2024-09-09,2024-10-01 12:15:24,HATS-14 b,313.215458,-25.687375,31,1
4,97409519,113.01,NaN,5,5,5,5,5,5,5,...,0.162580,1,2018-09-05,2020-10-27,2022-12-14 12:09:24,WASP-124,332.714333,-30.749719,9,2


In [59]:
# creating empty lists to be turned into columns
trans_num = []
trans_time = []

# looping over all rows in the dataset
for index, row in toi.iterrows():
    num, time = astroplan_init('Sutherland',
                               row['RA (deg)'],
                               row['Dec (deg)'],
                               row['TOI'],
                               row['Epoch (BJD)'],
                               row['Period (days)'],
                               row['Duration (hours)'],
                               '2025-02-12 16:00'
                               )
    trans_num.append(num)
    trans_time.append(time)

In [60]:
# copying the dataframe
toi_filtered = toi.copy()

# appending the new columns to the dataframe
toi_filtered.loc[:,'Num Transits'] = trans_num
toi_filtered.loc[:,'Transit Times'] = trans_time

# keeping only rows with at least one observable transit
toi_trans = toi_filtered.loc[toi_filtered['Num Transits'] >= 1]

,TIC ID,TOI,Previous CTOI,Master,SG1A,SG1B,SG2,SG3,SG4,SG5,...,Date TOI Alerted (UTC),Date TOI Updated (UTC),Date Modified,Comments,RA (deg),Dec (deg),Mookodi Comp,SHOC Comp,Num Transits,Transit Times
17,382188882,276.01,NaN,5,5,5,5,5,5,5,...,2018-11-30,2023-11-01,2023-11-03 12:13:21,TFOP FP/SB1,81.353083,-55.019925,4,2,1,2025-02-18 20:46:55;2025-02-18 22:46:44
33,349518800,407.01,NaN,5,5,5,5,3,5,5,...,2019-01-16,2023-10-02,2023-10-02 00:00:00,TFOP SB1/APC,111.211708,-62.092431,13,2,2,2025-02-12 22:27:12;2025-02-13 01:51:41;2025-0...
44,4616072,466.01,NaN,5,5,5,5,5,5,5,...,2019-02-22,2021-10-07,2022-12-14 12:09:24,HATS-45 b,101.994250,-21.910711,35,3,1,2025-02-23 19:23:59;2025-02-23 22:24:24
46,47911178,471.01,NaN,5,5,5,5,5,5,5,...,2019-02-27,2021-10-07,2022-12-14 12:09:24,WASP-101 b,98.351083,-23.486086,3,1,1,2025-02-22 17:50:00;2025-02-22 20:32:26
47,52640302,473.01,NaN,5,5,5,5,5,5,5,...,2019-02-22,2021-10-07,2022-12-14 12:09:24,WASP-64 b,101.114917,-32.858389,26,1,2,2025-02-12 20:08:59;2025-02-12 22:34:42;2025-0...


In [61]:
# saving to a new csv file
toi_trans.to_csv('Tables/TESS_TOI_28Jan2025_Mookodi_Transits.csv', index=False)

# reading in the new csv file
test = pd.read_csv('../Tables/TESS_TOI_28Jan2025_Mookodi_Transits.csv')
test.head()

,TIC ID,TOI,Previous CTOI,Master,SG1A,SG1B,SG2,SG3,SG4,SG5,...,Date TOI Alerted (UTC),Date TOI Updated (UTC),Date Modified,Comments,RA (deg),Dec (deg),Mookodi Comp,SHOC Comp,Num Transits,Transit Times
0,382188882,276.01,NaN,5,5,5,5,5,5,5,...,2018-11-30,2023-11-01,2023-11-03 12:13:21,TFOP FP/SB1,81.353083,-55.019925,4,2,1,2025-02-18 20:46:55;2025-02-18 22:46:44
1,349518800,407.01,NaN,5,5,5,5,3,5,5,...,2019-01-16,2023-10-02,2023-10-02 00:00:00,TFOP SB1/APC,111.211708,-62.092431,13,2,2,2025-02-12 22:27:12;2025-02-13 01:51:41;2025-0...
2,4616072,466.01,NaN,5,5,5,5,5,5,5,...,2019-02-22,2021-10-07,2022-12-14 12:09:24,HATS-45 b,101.994250,-21.910711,35,3,1,2025-02-23 19:23:59;2025-02-23 22:24:24
3,47911178,471.01,NaN,5,5,5,5,5,5,5,...,2019-02-27,2021-10-07,2022-12-14 12:09:24,WASP-101 b,98.351083,-23.486086,3,1,1,2025-02-22 17:50:00;2025-02-22 20:32:26
4,52640302,473.01,NaN,5,5,5,5,5,5,5,...,2019-02-22,2021-10-07,2022-12-14 12:09:24,WASP-64 b,101.114917,-32.858389,26,1,2,2025-02-12 20:08:59;2025-02-12 22:34:42;2025-0...


### Final lists
below is the code for the lists provided in the writeup

In [62]:
# first 10 potentially observable and number of transits for each
toi_trans[['TOI', 'Num Transits']].head(10)

,TOI,Num Transits
17,276.01,1
33,407.01,2
44,466.01,1
46,471.01,1
47,473.01,2
51,495.01,1
52,496.01,1
55,535.01,1
58,545.01,1
62,551.01,1


In [63]:
# list the number of TOIs with 1 observable transit
len(toi_trans[toi_trans['Num Transits'] == 1])

139

In [64]:
# list the number of TOIs with 2 observable
len(toi_trans[toi_trans['Num Transits'] == 2])

64

In [65]:
# list the number of TOIs with 3 observable
len(toi_trans[toi_trans['Num Transits'] == 3])

22

In [66]:
# list the number of TOIs with > 3 observable
len(toi_trans[toi_trans['Num Transits'] > 3])

29